# Why the processing steps matter: a model that learned the scanner

The steps in `ctkit`'s pipeline — reorient, clip, resample, standardize the size,
normalize — read like housekeeping. This notebook is the argument for them.

The setup is the one every multi-site imaging study is in: **two scanners, and a patient
population that is not the same on both**. The site with the newer scanner sees more of
the diseased patients. So anything that tells Scanner A's images apart from Scanner B's —
voxel size, calibration offset, reconstruction kernel, stored axis order, field of view —
is now correlated with the label, and a model is free to learn *the scanner* instead of
*the disease*. Preprocessing is where most of those shortcuts are taken away.

We train one small classifier several ways:

1. on the **raw volumes**, resized to a common grid the way people usually do it;
2. on volumes put through a **ctkit protocol**;
3. on the same protocol with **one step switched off at a time**.

Each is scored on two held-out sets: an **internal** one drawn from the same confounded
population as the training data, and an **external** one where both scanners are used for
diseased and healthy patients equally. The gap between those two numbers is the story.

The cohort is synthetic on purpose. It is the only way to know the ground truth about
which part of the image is biology and which part is the scanner, and it means the
notebook runs in a few minutes on a laptop with no GPU, no downloads, and no segmentation
model. [`quickstart.ipynb`](quickstart.ipynb) is the tour of the package on real data;
this notebook is the reason to bother.

In [ ]:
try:
    import ctkit
except ImportError:
    %pip install -q ctkit

import logging
import os
import time

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, zoom

import ctkit
from ctkit import Dataset, ProcessingConfig

# The ablation table below is easier to read without per-series log lines.
logging.getLogger("ctkit").setLevel(logging.ERROR)

ROOT = "data/confounded"
print("ctkit", ctkit.__version__)

## 1. Two scanners, one confound

Every row below is a real way two CT scanners differ, the shortcut it hands to a model,
and the step that takes the shortcut away.

| | Scanner A | Scanner B | What a model can read off it | Step that removes it |
| --- | --- | --- | --- | --- |
| in-plane voxel | 0.9 mm | 1.2 mm | a 10 mm lesion is 11 voxels wide on A and 8 on B, so "size in voxels" is partly a readout of the scanner | `resample` |
| slice thickness | 5.0 mm | 2.5 mm | the same, through-plane | `resample` |
| field of view / matrix | 72 × 72 × 24 | 64 × 64 × 48 | the array's *shape* identifies the scanner before a single voxel is looked at | `resample` + `standardize_size` |
| HU calibration | +45 HU | 0 HU | the mean intensity of the volume identifies the scanner | `normalize` |
| streak / metal artifact | 35% of scans | none | the maximum value identifies the scanner — and drags every statistic computed over the volume with it | `clip` |
| stored axis order | RAS | LPS | anatomy arrives mirrored, so the same finding lands in a different corner of the array | `orient` |
| reconstruction kernel | smooth | sharp | the noise texture identifies the scanner | **nothing below** — see §6 |

The last row is there to keep the notebook honest: preprocessing removes the leaks that
come from *geometry and intensity units*. It does not launder the image.

In [ ]:
SCANNERS = {
    "SiteA": dict(
        label="Scanner A (smooth kernel)",
        spacing=(0.9, 0.9, 5.0),   # mm
        shape=(72, 72, 24),
        hu_offset=45.0,            # calibration / contrast-phase shift
        noise=6.0,                 # HU
        blur=1.2,                  # smooth reconstruction kernel
        stored_lps=False,          # stored RAS
        metal_rate=0.35,           # fraction of scans with hyperdense streaks
    ),
    "SiteB": dict(
        label="Scanner B (sharp kernel)",
        spacing=(1.2, 1.2, 2.5),
        shape=(64, 64, 48),
        hu_offset=0.0,
        noise=14.0,
        blur=0.0,
        stored_lps=True,           # stored LPS: arrives mirrored
        metal_rate=0.0,
    ),
}

pd.DataFrame(SCANNERS).T

### The phantom

Each case is a body cross-section containing an organ, and inside the organ an eccentric
lesion. **The disease is the lesion**: in a diseased patient it is larger (≈10.5 mm vs
≈6 mm radius) and denser (≈+45 HU vs ≈+12 HU above the organ). That is the only signal a
model is supposed to use, and it is defined in millimetres and Hounsfield units — that is,
in the patient, not in the array.

Two numbers to keep in mind, because they are what makes this hard:

- the density difference between a diseased and a healthy lesion (≈33 HU) is *smaller*
  than the calibration offset between the two scanners (45 HU);
- the size difference between a diseased and a healthy lesion (≈4.5 mm) is comparable to
  what the difference in voxel size does to the same lesion measured in voxels.

So in raw arrays, "which scanner" and "which patient" move the same numbers by the same
amount, and the scanner is the more reliable of the two.

In [ ]:
def synthesize_case(rng, site, disease):
    """One scan and its label mask, as this scanner would have reconstructed it."""
    scanner = SCANNERS[site]
    shape, spacing = scanner["shape"], scanner["spacing"]

    # A grid in millimetres, centred on the image, so the anatomy below can be
    # specified in patient units and rendered onto either scanner's voxel grid.
    axes = [(np.arange(n) - (n - 1) / 2) * s for n, s in zip(shape, spacing)]
    x, y, z = np.meshgrid(*axes, indexing="ij")

    # ---- the patient: identical physics whichever scanner is used
    body_r = 30.0 + rng.normal(0, 1.5)
    organ_c = np.array([11.0, -5.0, 0.0]) + rng.normal(0, 1.5, 3)
    organ_r = 17.0 + rng.normal(0, 1.0)
    lesion_c = organ_c + np.array([6.0, 0.0, 0.0]) + rng.normal(0, 1.0, 3)
    lesion_r = (10.5 if disease else 6.0) + rng.normal(0, 1.2)
    lesion_hu = (45.0 if disease else 12.0) + rng.normal(0, 10.0)

    body = np.sqrt(x ** 2 + y ** 2) < body_r
    organ = np.sqrt((x - organ_c[0]) ** 2 + (y - organ_c[1]) ** 2 + (z - organ_c[2]) ** 2) < organ_r
    lesion = np.sqrt((x - lesion_c[0]) ** 2 + (y - lesion_c[1]) ** 2 + (z - lesion_c[2]) ** 2) < lesion_r

    data = np.full(shape, -1000.0, dtype=np.float32)   # air
    data[body] = 40.0
    data[organ] = 60.0
    data[lesion] = 60.0 + lesion_hu

    mask = np.zeros(shape, dtype=np.uint8)
    mask[organ] = 1       # ctkit's convention: 1 = organ, 2 = tumour
    mask[lesion] = 2

    # ---- the scanner: noise, kernel, calibration, artefacts
    data = data + rng.normal(0, scanner["noise"], shape)
    if scanner["blur"]:
        data = gaussian_filter(data, scanner["blur"])
    data += scanner["hu_offset"]
    if rng.random() < scanner["metal_rate"]:
        for _ in range(rng.integers(3, 8)):
            i, j, k = (rng.integers(0, n) for n in shape)
            data[max(0, i - 1):i + 2, max(0, j - 1):j + 2, k] = rng.uniform(1800, 3000)

    # ---- how the archive stores it. A negative affine entry means the array is
    # held in the opposite direction, so the anatomy comes back mirrored until
    # something reorients it.
    sx, sy, sz = spacing
    if scanner["stored_lps"]:
        data, mask = data[::-1, ::-1, :], mask[::-1, ::-1, :]
        sx, sy = -sx, -sy
    affine = np.diag([sx, sy, sz, 1.0])
    affine[:3, 3] = -np.array([sx, sy, sz]) * (np.array(shape) - 1) / 2
    return nib.Nifti1Image(data, affine), nib.Nifti1Image(mask, affine)

### The splits

| split | Scanner A | Scanner B | what it is |
| --- | --- | --- | --- |
| `train` | 50 scans, 42 diseased | 50 scans, 8 diseased | the confounded study |
| `test_int` | 40 scans, 34 diseased | 40 scans, 6 diseased | **internal** validation: a held-out slice of the same study |
| `test_ext` | 60 scans, 30 diseased | 60 scans, 30 diseased | **external** validation: another hospital, where both scanners are used for everyone |

Every split is 50% diseased overall, so **0.50 is chance** everywhere. Two more numbers
worth writing down before running anything:

- the rule *"call it disease if it came from Scanner A"* scores **0.85** on the internal
  test set and **0.50** on the external one. A model that has learned only the scanner
  will land near those two numbers.
- a model that has learned the lesion should score about the same on both.

In [ ]:
SPLITS = {  # split -> (scans per site, diseased per site)
    "train":    (50, {"SiteA": 42, "SiteB": 8}),
    "test_int": (40, {"SiteA": 34, "SiteB": 6}),
    "test_ext": (60, {"SiteA": 30, "SiteB": 30}),
}


def build_cohort(root=ROOT, seed=0):
    """Write the cohort as NIfTI, one directory per case, and return the label table."""
    rng = np.random.default_rng(seed)
    rows = []
    for split, (n, n_diseased) in SPLITS.items():
        for site in SCANNERS:
            draw = np.array([1] * n_diseased[site] + [0] * (n - n_diseased[site]))
            rng.shuffle(draw)
            for i, disease in enumerate(draw):
                image, mask = synthesize_case(rng, site, int(disease))
                case = f"{site}_{i:03d}"
                out = os.path.join(root, split, case)
                os.makedirs(out, exist_ok=True)
                nib.save(image, os.path.join(out, "imaging.nii.gz"))
                nib.save(mask, os.path.join(out, "segmentation.nii.gz"))
                rows.append(dict(split=split, series_id=case, site=site,
                                 scanner=SCANNERS[site]["label"], disease=int(disease)))
    labels = pd.DataFrame(rows)
    labels.to_csv(os.path.join(root, "labels.csv"), index=False)
    return labels


start = time.time()
labels = build_cohort()
print(f"{len(labels)} scans written in {time.time() - start:.0f}s")

pd.crosstab([labels.split, labels.site], labels.disease, rownames=["split", "site"], colnames=["disease"])

## 2. The confound is visible before you train anything

Nothing so far required a model. Load the cohort as a `Dataset` and look at what each
series announces about itself.

In [ ]:
def describe(split):
    rows = []
    for image in Dataset(f"{ROOT}/{split}"):
        rows.append(dict(
            series_id=image.series_id,
            shape=image.shape,
            spacing=tuple(round(float(s), 2) for s in image.spacing),
            orientation=image.orientation,
            mean_hu=round(float(image.array.mean())),
            max_hu=round(float(image.array.max())),
        ))
        image.unload()
    frame = pd.DataFrame(rows)
    return frame.merge(labels[labels.split == split], on="series_id")


train_facts = describe("train")
train_facts.groupby("site").agg(
    n=("series_id", "size"),
    shape=("shape", lambda s: s.iloc[0]),
    spacing=("spacing", lambda s: s.iloc[0]),
    orientation=("orientation", lambda s: s.iloc[0]),
    mean_hu_min=("mean_hu", "min"),
    mean_hu_max=("mean_hu", "max"),
    max_hu=("max_hu", "max"),
    prevalence=("disease", "mean"),
)

`shape`, `spacing` and `orientation` identify the scanner exactly — they are constant
within a site and different between them — and the ranges of `mean_hu` and `max_hu` barely
overlap. That is what makes this a confound rather than a nuisance: the scanner is fully
determined by the image, and the scanner predicts the label 84% of the time in the
training set.

A one-line "model" makes the point without looking at anatomy at all:

In [ ]:
threshold = train_facts.mean_hu.median()
stump = (train_facts.mean_hu > threshold).astype(int)   # a single split on mean intensity

print(f"identifies the scanner: {(stump == (train_facts.site == 'SiteA')).mean():.2f}")
print(f"predicts the diagnosis: {(stump == train_facts.disease).mean():.2f}")

A single threshold on average brightness recovers the scanner 98% of the time, and
therefore the diagnosis 82% of the time. A model that never learns anything beyond "how
bright is this volume" is already most of the way to the reported accuracy — and will stay
there for exactly as long as the scanner keeps correlating with the label.

Side by side, the same anatomy on the two scanners. Note the different array shapes and
the mirrored organ, both visible before the intensities are even considered.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, site in zip(axes, SCANNERS):
    image = Dataset(f"{ROOT}/train").get(f"{site}_000")
    image.plot(window=(-200, 300), ax=ax, overlay_mask=False)
    ax.set_title(f"{SCANNERS[site]['label']}\n{image.shape}  {image.orientation}  "
                 f"mean {image.array.mean():.0f} HU")
fig.suptitle("Raw, as the archive stores it", y=1.02)
fig.tight_layout()

## 3. Train on the raw volumes

The usual naive pipeline: load the array, rescale it to a fixed grid so the shapes match,
z-score the features, fit a classifier.

The model is a ridge-regression linear probe on a 16 × 16 × 8 downsampling of the volume.
It is deliberately dull, and it is solved in closed form — no learning rate, no stopping
rule — so that every arm below is compared on *the data it was given*, not on how long
someone let an optimizer run. The regularization strength is fixed once, here, and is
never tuned per arm.

In [ ]:
GRID = (16, 16, 8)   # the model's input resolution
L2 = 1.0             # ridge strength, fixed for every arm in this notebook


def to_tensor(arrays):
    """Every volume onto one small grid, flattened. Naive rescaling, on purpose:
    this is the step that stands in for whatever a training script does when the
    shapes in a cohort do not match."""
    return np.stack([
        zoom(np.asarray(a, dtype=np.float32),
             [g / s for g, s in zip(GRID, a.shape)], order=1).ravel()
        for a in arrays
    ])


def fit_ridge(X, y, l2=L2):
    """Ridge regression on +/-1 targets, solved exactly through the dual."""
    t = 2.0 * np.asarray(y, dtype=float) - 1.0
    intercept, n = t.mean(), len(t)
    dual = np.linalg.solve(X @ X.T + l2 * n * np.eye(n), t - intercept)
    return X.T @ dual, intercept


def accuracy(model, X, y):
    weights, intercept = model
    return float((((X @ weights + intercept) > 0).astype(int) == np.asarray(y)).mean())

In [ ]:
def load_split(split, config=None):
    """Volumes for one split, raw or run through `config`, with their labels."""
    data = Dataset(f"{ROOT}/{split}")
    if config is None:
        arrays, series_ids = [], []
        for image in data:
            arrays.append(image.array.copy())
            series_ids.append(image.series_id)
            image.unload()
    else:
        processed = data.process(config, progress=False)
        arrays = [image.array for image in processed]
        series_ids = processed.series_ids

    rows = labels[labels.split == split].set_index("series_id").loc[series_ids]
    return arrays, rows.disease.to_numpy(), (rows.site == "SiteA").astype(int).to_numpy()


CACHE = {}


def run_arm(name, config):
    """Fit on train, score on both test sets, and ask whether the model is
    answering the diagnosis question or the scanner question."""
    if config is not None:
        # The protocol is fixed on the training cohort and then applied unchanged
        # to both test sets -- target_shape is a property of the study, not
        # something each split gets to re-measure.
        config = Dataset(f"{ROOT}/train").resolve_config(config, progress=False)

    key = repr(config)
    if key not in CACHE:
        CACHE[key] = {split: load_split(split, config) for split in SPLITS}
    data = CACHE[key]

    X = {split: to_tensor(arrays) for split, (arrays, _, _) in data.items()}
    mean, sd = X["train"].mean(0), X["train"].std(0) + 1e-6
    X = {split: (values - mean) / sd for split, values in X.items()}

    model = fit_ridge(X["train"], data["train"][1])
    external_site = data["test_ext"][2]
    return {
        "arm": name,
        "train": accuracy(model, X["train"], data["train"][1]),
        "internal": accuracy(model, X["test_int"], data["test_int"][1]),
        "external": accuracy(model, X["test_ext"], data["test_ext"][1]),
        # On the external split, site and diagnosis are independent. Anything left
        # of the agreement between the model's call and the site is the model
        # reading the scanner.
        "agrees with scanner": max(accuracy(model, X["test_ext"], external_site),
                                   1 - accuracy(model, X["test_ext"], external_site)),
    }


raw = run_arm("raw (no preprocessing)", None)
pd.Series(raw).to_frame("raw").T

Read that row twice.

- **internal 0.89.** In a paper, with a held-out test set from the same study, this is a
  good result and there is nothing in it that looks wrong.
- **external 0.58.** At the hospital where the two scanners are used for everyone, the
  model is barely better than a coin.
- **agrees with scanner 0.88.** On the external set, where the scanner says *nothing*
  about the diagnosis, the model's answer still matches the scanner almost 9 times in 10.
  It is not diagnosing. It is identifying the machine.

Note also that internal accuracy (0.89) is barely above the 0.85 that the rule "disease if
Scanner A" would score. The internal test set cannot tell the two apart, because the
shortcut is just as available there as it was in training. **A held-out split drawn from a
confounded cohort is confounded too.**

## 4. Train on the processed volumes

The same data, the same model, through a `ctkit` protocol. Five steps, each closing one
row of the table in §1: **one axis order, one intensity range, one voxel grid, one array
shape, one intensity scale.**

Segmentation and the ROI crop are switched off. They are the other half of what `ctkit`
does, but they are about *where to look*, not about the scanner, and mixing them in would
make the ablation below harder to read. They come back as one extra row at the end.

The protocol is an object: it prints, it saves to YAML, and — the part that matters here —
it is fixed on the training cohort and then applied unchanged to both test sets.

In [ ]:
PROTOCOL = ProcessingConfig(
    orient=True, target_orientation="RAS",       # one axis order
    clip=True, clip_min=-200, clip_max=300,      # one intensity range
    resample=True, target_spacing=(1.0, 1.0, 3.0),   # one voxel grid
    standardize_size=True, target_shape=None,    # one array shape (None: measure it)
    normalize=True, normalization_method="volume",   # one intensity scale
    segment=False, mask=False,   # the ROI steps are a separate question -- see below
)

print(PROTOCOL.describe())

In [ ]:
# target_shape was left unset, so the cohort measures it: the 95th-percentile shape
# after every earlier step has run. It is measured on the training cohort and then
# frozen, which is what makes the same protocol applicable to the test sets.
resolved = Dataset(f"{ROOT}/train").resolve_config(PROTOCOL, progress=False)
print("target_shape measured from the training cohort:", resolved.target_shape)

processed = run_arm("full ctkit protocol", resolved)
pd.DataFrame([raw, processed]).set_index("arm")

Internal accuracy barely moved (0.89 → 0.94) — from inside the study, preprocessing looks
like it did nothing at all. External accuracy went from **0.58 to 0.96**, and the model's
agreement with the scanner fell from 0.88 to **0.51**, which is chance: on the external set
the processed model's answers now have nothing to do with which machine took the picture.

The processed model scores the same on both test sets, which is what a model that learned
the lesion should do. The raw model was not a worse model of the disease — it was never
modelling the disease at all.

Here is what the protocol did to the two scans from §2:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, site in zip(axes, SCANNERS):
    image = Dataset(f"{ROOT}/train").get(f"{site}_000").process(resolved)
    ax.imshow(np.rot90(image.array[:, :, image.shape[2] // 2]), cmap="gray")
    ax.set_title(f"{SCANNERS[site]['label']}\n{image.shape}  {image.orientation}  "
                 f"mean {image.array.mean():.2f} (z-scored)")
    ax.axis("off")
fig.suptitle("After the protocol: one voxel grid, one orientation, one intensity scale", y=1.02)
fig.tight_layout()

Same array shape, same orientation, same voxel size, same intensity scale. Every column of
the table in §2 that used to identify the site — `shape`, `spacing`, `orientation`, and the
range the intensities live in — now reads the same on both, and what is left in the array
is anatomy, which is what the label was supposed to be about.

## 5. Which step was doing the work

One row per step, with that step switched off and everything else left alone. `ctkit`
protocols are dataclasses, so an ablation is `PROTOCOL.replace(step=False)`.

This takes a couple of minutes: each arm re-processes all 300 scans, plus one extra pass
over the training cohort to measure the output size that arm needs.

In [ ]:
ABLATIONS = [
    ("no orient",           PROTOCOL.replace(orient=False)),
    ("no clip",             PROTOCOL.replace(clip=False)),
    ("no resample",         PROTOCOL.replace(resample=False)),
    ("no standardize_size", PROTOCOL.replace(standardize_size=False)),
    ("no normalize",        PROTOCOL.replace(normalize=False)),
    # and the step held out of the protocol above, added back in
    ("+ ROI mask",          PROTOCOL.replace(mask=True, crop_to_mask=True, crop_padding=5)),
]

start = time.time()
results = pd.DataFrame(
    [raw, processed] + [run_arm(name, config) for name, config in ABLATIONS]
).set_index("arm")

# 120 external cases, so differences smaller than about 0.06 are not differences.
results["external +/-"] = np.sqrt(results.external * (1 - results.external) / 120).round(3)
print(f"{time.time() - start:.0f}s")
results.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
order = results.index[::-1]
y = np.arange(len(order))

ax.barh(y + 0.2, results.loc[order, "internal"], height=0.38,
        color="#b9c6d6", label="internal validation (same study)")
ax.barh(y - 0.2, results.loc[order, "external"], height=0.38,
        xerr=results.loc[order, "external +/-"], color="#2f5d8a", ecolor="#33383d",
        label="external validation (confound broken)")
ax.axvline(0.5, color="#33383d", lw=1, ls="--")
ax.text(0.5, len(order) - 0.35, " chance", fontsize=9, color="#33383d", va="center")

ax.set_yticks(y, order)
ax.set_xlabel("accuracy")
ax.set_xlim(0, 1.0)
ax.legend(loc="lower right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Internal validation cannot tell these apart. External validation can.")
fig.tight_layout()

Two things to take from the figure.

**The light bars are all the same.** Every arm lands between 0.84 and 0.94 on the internal
test set, the raw volumes included. If internal validation is the only number reported,
none of these preprocessing decisions appears to matter — which is exactly why they get
skipped.

**The dark bars are not.** Every step that was closing a leak costs external accuracy when
it is switched off, and every one of them raises the model's agreement with the scanner:

- **`clip`** (0.96 → 0.51, scanner agreement 0.96) — Scanner A's streak artefacts reach
  3000 HU. They tag the scanner on their own, and because they are extreme they drag the
  per-volume mean and standard deviation with them, which corrupts the normalization that
  runs afterwards. The step that looks like contrast adjustment is doing outlier control,
  and without it this arm is the worst in the table — worse than no preprocessing at all.
- **`orient`** (0.96 → 0.63) and **`normalize`** (0.96 → 0.63) — Site B's scans arrive
  mirrored, so the same lesion lands on the opposite side of the array; Site A's scans sit
  45 HU higher than Site B's, a per-scan constant that z-scoring removes and nothing else
  does.
- **`resample`** (0.96 → 0.64) — without a common voxel grid, a millimetre of anatomy is a
  different number of voxels on each scanner, so lesion size measured in voxels is part
  biology and part machine. This is the least visible failure of the lot: nothing about the
  arrays looks wrong.
- **`standardize_size`** (0.96 → 0.80) — with the shapes left unequal, the training script
  still has to make them match, and it does it by rescaling each volume onto the model's
  input grid. That rescaling factor is site-dependent, so it destroys physical size and
  puts the field-of-view difference straight back into the tensor.

**The last row is the honest one.** Adding the ROI crop back (`+ ROI mask`, 0.89) does not
help here, and slightly hurts. It is not a debiasing step: it exists so that a cohort of
full-resolution volumes fits in memory and so that a model looks at the organ rather than
at whatever else is in the field of view. In this phantom there is nothing else in the
field of view, so all it does is throw away context. Not every step in a pipeline is there
for the same reason, and this experiment can only measure the ones that are about the
confound.

One thing the table cannot tell you: on 120 external cases, differences smaller than
roughly 0.1 are inside the noise, so `no orient` (0.63), `no normalize` (0.63) and
`no resample` (0.64) are tied as far as this cohort is concerned — the ordering among them
is not a result. The separations that carry the argument — each ablation against the full
protocol, and the full protocol against raw — are two to four times larger than that.

## 6. What preprocessing does not fix

It would be a nice ending if the processed volumes no longer carried any trace of the
scanner. They do. Ask the same probe to predict the *site* instead of the diagnosis:

In [ ]:
def site_probe(config):
    data = {split: load_split(split, config) for split in ("train", "test_ext")}
    X = {split: to_tensor(arrays) for split, (arrays, _, _) in data.items()}
    mean, sd = X["train"].mean(0), X["train"].std(0) + 1e-6
    X = {split: (values - mean) / sd for split, values in X.items()}
    model = fit_ridge(X["train"], data["train"][2])
    return accuracy(model, X["test_ext"], data["test_ext"][2])


print(f"site recoverable from raw volumes:       {site_probe(None):.2f}")
print(f"site recoverable from processed volumes: {site_probe(resolved):.2f}")

Still essentially perfect. The reconstruction kernel is left in the image — smooth versus
sharp is a noise texture, and no amount of reorienting, resampling or z-scoring removes
it. Preprocessing removed the leaks that live in *geometry and intensity units*, which is
what made the shortcut easier to learn than the lesion. It did not make the scans
indistinguishable, and nothing in a preprocessing library will.

So the steps above are necessary and not sufficient. The rest of the job is study design:

- **Report external validation, on scanners the model has not seen.** It is the only
  number in this notebook that ever showed a problem.
- **Check what the confound actually is.** `ctkit.check` and `Dataset.filter` produce the
  pass/fail table with the measurements behind each decision, and
  `Dataset.check_consistency` flags series whose intensity distribution is unlike the rest
  of the cohort — a different contrast phase, a failed rescale, a scanner that does not
  belong.
- **Stratify the splits by site**, so that internal validation is not measuring the
  shortcut it was supposed to test for.
- **Balance or model the confound** — match prevalence across scanners where the data
  allows it, or include site as a covariate and check that the imaging signal survives it.
- **Augment across the residual differences** you cannot remove: noise, blur, kernel.

One caveat on the numbers themselves. How badly the raw model collapses depends on how
much capacity it has: with the regularization turned far down, the raw model has room to
fit the lesion *and* the scanner and it looks better externally, and turned up, it leans on
the shortcut harder and looks worse. The direction never changes, but the size does. Which
is the argument, really — whether your model exploits a confound is not something you get
to find out from the model. Take the shortcut out of the data.

## 7. Reproducing it

The protocol that produced the processed arm is one serializable object. `process()`
writes it next to the output, so the exact preprocessing behind a result travels with the
result — including for whoever tries to reproduce your external validation.

In [ ]:
resolved.to_yaml("data/protocol.yaml")
print(open("data/protocol.yaml").read())

In [ ]:
# Reproducing the processed cohort, from the config alone.
rebuilt = Dataset(f"{ROOT}/test_ext").process("data/protocol.yaml", out_dir="data/rebuilt",
                                              progress=False)
print(len(rebuilt), "scans,", rebuilt[0].shape)

## Summary

| | internal | external | agrees with scanner |
| --- | --- | --- | --- |
| raw volumes | 0.89 | 0.58 | 0.88 |
| ctkit protocol | 0.94 | 0.96 | 0.51 |

The raw-data model was not a bad model. On every number a confounded study produces, it
was a good one. It answered a different question than the one it was asked, and internal
validation had no way of noticing.

The steps that mattered were the ones that put every scan on the same voxel grid, the same
array shape, the same orientation and the same intensity scale. None of them is
interesting on its own, and none of them showed up in internal validation. Together they
were the difference between a model that reads anatomy and a model that reads the machine.

### Next

- [`quickstart.ipynb`](quickstart.ipynb) — the same pipeline on a real TCIA collection
- `Dataset.filter` / `ctkit.check` — the quality-control table, and the other half of this
  problem: series that should never have entered the cohort
- `ProcessingConfig.for_dataset` — the curated protocol for a named collection